In [1]:
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

In [2]:
ROOT = Path("/home/hanwenying")
W76DIR = ROOT / "rothman-sam/w76"
REFDIR = ROOT / "rothman-sam/ref"

In [131]:
def formatattributes(attributestr):
	attr_dict = {}
	attributestr = attributestr.split(';')

	for i in range(len(attributestr)):
		attr = attributestr[i].strip().split('""')
		try:
			attr_dict[attr[0].strip()] = attr[1].strip()
		except IndexError:
			continue
	
	return attr_dict

In [155]:
x = 'gene_id "CELE_C41D11.1"; transcript_id ""; db_xref "GeneID:172050"; db_xref "WormBase:WBGene00016561"; gbkey "Gene"; gene "eri-6"; gene_biotype "protein_coding"; locus_tag "CELE_C41D11.1"; partial "true"; '

In [158]:
def formatattributes(attributestr):
	attributestr = attributestr.split(';')
	d = {}
	for i in attributestr:
		i = i.strip()
		i = i.split(' ')
		try:
			d[i[0]] = i[1]
		except IndexError:
			continue
	
	return d

In [188]:
ex = pd.read_csv(W76DIR / "affectedgenes" / "A1.bam_affectedgenes.tsv", sep='\t')
ex.shape

(55, 13)

In [189]:
ex['affectedgene'] = ex['affectedgene'].apply(formatattributes)

In [194]:
bams = ['A1.bam', 'B2.bam', 'C3.bam', 'D4.bam', 'E5.bam', 'F6.bam', 'G7.bam', 'H8.bam', 'I9.bam', 'J10.bam', 'K11.bam', 'L12.bam', 'M13.bam', 'N14.bam', 'O15.bam', 'P16.bam', 'Q17.bam']
chroms = ['NC_001328.1', 'NC_003279.8', 'NC_003280.10', 'NC_003281.10', 'NC_003282.8', 'NC_003283.11', 'NC_003284.9']

In [195]:
maxindels = pd.DataFrame()

In [196]:
for bam in bams:
	try:
		indels = pd.read_csv(W76DIR / "affectedgenes" / f"{bam}_affectedgenes.tsv", sep='\t')
		bam = bam.split('.')[0]
		indels['affectedgene'] = indels['affectedgene'].apply(formatattributes)
		indels = indels.join(indels['affectedgene'].apply(pd.Series)).drop(columns=['affectedgene'])
		indels['bam'] = bam

		maxindels = pd.concat([maxindels, indels])
	except:
		continue

In [197]:
maxindels

,>indel_type,call_type,chr,sttpos,endpos,indel_length,indel_str,#indel_depth,#ttl_depth,details(indelcall_indeltype_depth),...,partial,note,product,standard_name,transcript_biotype,exon_number,protein_id,bam,pseudo,pseudogene
0,DEL,Homo,NC_003279,4466741,4466828,88,TTTATATTATCCCCCGTATGTTAATCATCAGAAAAGTACAATAATT...,6,6,DEL_B_6_4466740,...,"""true""",NaN,NaN,NaN,NaN,NaN,NaN,A1,NaN,NaN
1,DEL,Homo,NC_003279,4466741,4466828,88,TTTATATTATCCCCCGTATGTTAATCATCAGAAAAGTACAATAATT...,6,6,DEL_B_6_4466740,...,NaN,"""Product","""Enhanced","""C41D11.1d.1""","""mRNA""",NaN,NaN,A1,NaN,NaN
2,DEL,Homo,NC_003279,4466741,4466828,88,TTTATATTATCCCCCGTATGTTAATCATCAGAAAAGTACAATAATT...,6,6,DEL_B_6_4466740,...,NaN,"""Product","""Enhanced","""C41D11.1f.1""","""mRNA""",NaN,NaN,A1,NaN,NaN
3,DEL,Homo,NC_003279,4466741,4466828,88,TTTATATTATCCCCCGTATGTTAATCATCAGAAAAGTACAATAATT...,6,6,DEL_B_6_4466740,...,NaN,"""Product","""Enhanced","""C41D11.1f.1""","""mRNA""","""3""",NaN,A1,NaN,NaN
4,DEL,Homo,NC_003280,950444,950448,5,TGACA,7,7,DEL_SID_7_950443,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,DEL,Hete,NC_003284,13354694,13354695,2,TG,6,9,DEL_SID_6_13354693,...,"""true""",NaN,"""WW","""Y40B1A.3b.1""","""mRNA""",NaN,NaN,Q17,NaN,NaN
112,DEL,Homo,NC_003284,13489606,13489607,2,TT,9,11,DEL_SID_9_13489605,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Q17,NaN,NaN
113,DEL,Homo,NC_003284,13489606,13489607,2,TT,9,11,DEL_SID_9_13489605,...,NaN,NaN,"""S-formylglutathione","""Y48G10A.1.1""","""mRNA""",NaN,NaN,Q17,NaN,NaN
114,DEL,Homo,NC_003284,13489606,13489607,2,TT,9,11,DEL_SID_9_13489605,...,NaN,NaN,"""S-formylglutathione","""Y48G10A.1.1""","""mRNA""","""1""",NaN,Q17,NaN,NaN


In [198]:
uniquebamgenes = maxindels[['gene_id', 'bam']].drop_duplicates()

In [269]:
lowgroup = ['A1', 'B2', 'C3', 'D4', 'E5', 'F6', 'G7', 'H8', 'Q17']
highgroup = ['I9', 'J10', 'K11', 'L12', 'M13', 'N14', 'O15', 'P16']

In [270]:
lowgenes = uniquebamgenes[uniquebamgenes['bam'].isin(lowgroup)]
highgenes = uniquebamgenes[uniquebamgenes['bam'].isin(highgroup)]

In [271]:
limitbams = len(lowgroup) - 6 # want to see at least [limitbams] bams for a deletion

In [272]:
lowgene_valid = pd.DataFrame(columns=['gene','bams'])
for gene in lowgenes['gene_id'].unique():
	numbams = len(lowgenes[lowgenes['gene_id'] == gene])
	if numbams >= limitbams:
		lowgene_valid.loc[len(lowgene_valid)] = {'gene':gene, 'bams':lowgenes[lowgenes['gene_id']==gene]['bam'].unique()}

In [273]:
set(lowgene_valid['gene']) - set(highgenes['gene_id'].tolist())

{'"CELE_C44E4.1"',
 '"CELE_F32B5.6"',
 '"CELE_F52B5.1"',
 '"CELE_Y34D9B.1"',
 '"CELE_Y40B1A.3"',
 '"CELE_Y40B1A.4"'}

In [274]:
lowgene_valid

,gene,bams
0,"""CELE_C41D11.1""","[A1, C3, E5, H8, Q17]"
1,"""CELE_Y34D9B.1""","[A1, B2, D4]"
2,"""CELE_F10G8.7""","[A1, B2, E5, H8, Q17]"
3,"""CELE_F32B5.6""","[A1, H8, Q17]"
4,"""CELE_C54C8.4""","[A1, B2, E5, F6, G7, H8, Q17]"
5,"""CELE_Y40B1A.4""","[A1, B2, C3]"
6,"""CELE_F46F11.8""","[A1, C3, D4, E5, F6, G7, H8, Q17]"
7,"""CELE_Y74C9A.3""","[A1, B2, D4, F6, G7, H8, Q17]"
8,"""CELE_Y92H12A.5""","[A1, B2, D4, E5, F6, G7, H8, Q17]"
9,"""CELE_F28B3.4""","[A1, B2, C3, D4, E5, G7]"


In [268]:
maxindels[maxindels['gene_id'] == '"CELE_Y40B1A.4"'].to_csv(W76DIR / "a1c3overlap.tsv", sep='\t', index=False)